# **Gold.Product**

In [0]:
use catalog smart_odoo;
drop table if exists gold.dim_product;
CREATE TABLE IF NOT EXISTS gold.dim_product
(
    product_id INT,
    categ_id INT,

    product_sku STRING,
    product_name STRING,
    product_type STRING,
    product_status STRING,

    product_list_price DECIMAL(18,2),
    product_cost DECIMAL(18,2),
    product_gross_margin_estimated DECIMAL(18,2),

    weight DECIMAL(18,2),
    volume DECIMAL(18,2),

    sale_ok BOOLEAN,
    purchase_ok BOOLEAN,

    categ_name STRING,
    category_full_name STRING,

    created_at DATE,
    updated_at DATE,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

MERGE INTO gold.dim_product AS target
USING
(
    SELECT
        CAST(pp.product_id AS int) AS product_id,
        CAST(pc.categ_id AS int) AS categ_id,

        CAST(pp.default_code AS string) AS product_sku,
        CAST(pp.product_tmpl_name AS string) AS product_name,
        pt.type AS product_type,

        CASE
            WHEN pp.active THEN 'Active'
            ELSE 'Inactive'
        END AS product_status,

        pt.list_price AS product_list_price,

        COALESCE(pp.standard_price, 0) AS product_cost,

        (
            pt.list_price - COALESCE(pp.standard_price, 0)
        ) AS product_gross_margin_estimated,

        pp.weight,
        pp.volume,

        pt.sale_ok,
        pt.purchase_ok,

        pc.name AS categ_name,
        pc.complete_name AS category_full_name,

        CAST(pt.created_at AS DATE) AS created_at,
        CAST(pt.updated_at AS DATE) AS updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.product_product pp

    JOIN silver.product_template pt
        ON pp.product_tmpl_id = pt.product_tmpl_id

    LEFT JOIN silver.product_category pc
        ON pt.categ_id = pc.categ_id

    WHERE pt.updated_at >=
    (
        SELECT COALESCE(MAX(updated_at), TIMESTAMP('1900-01-01'))
        FROM gold.dim_product
    )

) AS source

ON target.product_id = source.product_id

WHEN MATCHED
AND target.updated_at < source.updated_at

THEN UPDATE SET
    target.categ_id = source.categ_id,
    target.product_sku = source.product_sku,
    target.product_name = source.product_name,
    target.product_type = source.product_type,
    target.product_status = source.product_status,
    target.product_list_price = source.product_list_price,
    target.product_cost = source.product_cost,
    target.product_gross_margin_estimated = source.product_gross_margin_estimated,
    target.weight = source.weight,
    target.volume = source.volume,
    target.sale_ok = source.sale_ok,
    target.purchase_ok = source.purchase_ok,
    target.categ_name = source.categ_name,
    target.category_full_name = source.category_full_name,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT
(
    product_id,
    categ_id,
    product_sku,
    product_name,
    product_type,
    product_status,
    product_list_price,
    product_cost,
    product_gross_margin_estimated,
    weight,
    volume,
    sale_ok,
    purchase_ok,
    categ_name,
    category_full_name,
    created_at,
    updated_at,
    dwh_loaded_at
)

VALUES
(
    source.product_id,
    source.categ_id,
    source.product_sku,
    source.product_name,
    source.product_type,
    source.product_status,
    source.product_list_price,
    source.product_cost,
    source.product_gross_margin_estimated,
    source.weight,
    source.volume,
    source.sale_ok,
    source.purchase_ok,
    source.categ_name,
    source.category_full_name,
    source.created_at,
    source.updated_at,
    current_timestamp()
);

In [0]:
USE CATALOG smart_odoo;

CREATE TABLE IF NOT EXISTS gold.dim_product_combo
(
    combo_item_id INT,
    combo_id INT,
    product_id INT,
    company_id INT,

    combo_name STRING,
    extra_price DECIMAL(18,2),

    created_at DATE,
    updated_at DATE,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

MERGE INTO gold.dim_product_combo AS target

USING
(
    SELECT
        cast(ci.combo_item_id AS INT),
        cast(c.combo_id AS INT),
        cast(ci.product_id AS INT),
        cast(c.company_id AS INT),

        c.name AS combo_name,
        ci.extra_price,

        CAST(c.created_at AS DATE) AS created_at,
        CAST(c.updated_at AS DATE) AS updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.product_combo c

    JOIN silver.product_combo_item ci
        ON c.combo_id = ci.combo_id

    WHERE c.updated_at >=
    (
        SELECT COALESCE(MAX(updated_at), TIMESTAMP('1900-01-01'))
        FROM gold.dim_product_combo
    )

) source

ON target.combo_item_id = source.combo_item_id

WHEN MATCHED
AND target.updated_at < source.updated_at

THEN UPDATE SET
    target.combo_id = source.combo_id,
    target.product_id = source.product_id,
    target.company_id = source.company_id,
    target.combo_name = source.combo_name,
    target.extra_price = source.extra_price,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT
(
    combo_item_id,
    combo_id,
    product_id,
    company_id,
    combo_name,
    extra_price,
    created_at,
    updated_at,
    dwh_loaded_at
)

VALUES
(
    source.combo_item_id,
    source.combo_id,
    source.product_id,
    source.company_id,
    source.combo_name,
    source.extra_price,
    source.created_at,
    source.updated_at,
    current_timestamp()
);

In [0]:
CREATE TABLE IF NOT EXISTS gold.dim_product_pricelist
(
    pricelist_item_id INT,
    pricelist_id INT,
    company_id INT,

    product_tmpl_id INT,
    product_id INT,
    categ_id INT,

    applied_on STRING,
    compute_price STRING,

    fixed_price DECIMAL(18,2),
    percent_price DECIMAL(18,2),
    price_discount DECIMAL(18,2),
    price_surcharge DECIMAL(18,2),
    price_markup DECIMAL(18,2),
    price_min_margin DECIMAL(18,2),
    price_max_margin DECIMAL(18,2),

    valid_from TIMESTAMP,
    valid_to TIMESTAMP,

    created_at TIMESTAMP,
    updated_at TIMESTAMP,

    dwh_loaded_at TIMESTAMP
)
USING DELTA;

MERGE INTO gold.dim_product_pricelist AS target

USING
(
    SELECT
        cast(pli.pricelist_item_id AS INT),
        cast(pli.pricelist_id AS INT),
        cast(pli.company_id AS INT),

        cast(pli.product_tmpl_id AS INT),
        cast(pli.product_id AS INT),
        cast(pli.categ_id AS INT),

        pli.applied_on,
        pli.compute_price,

        pli.fixed_price,
        pli.percent_price,
        pli.price_discount,
        pli.price_surcharge,
        pli.price_markup,
        pli.price_min_margin,
        pli.price_max_margin,

        cast(pli.date_start  AS DATE) AS valid_from,
        cast(pli.date_end AS DATE) AS valid_to,

        pl.created_at,
        pl.updated_at,

        current_timestamp() AS dwh_loaded_at

    FROM silver.product_pricelist_item pli

    JOIN silver.product_pricelist pl
        ON pli.pricelist_id = pl.pricelist_id

    WHERE pl.updated_at >=
    (
        SELECT COALESCE(MAX(updated_at), TIMESTAMP('1900-01-01'))
        FROM gold.dim_product_pricelist
    )

) source

ON target.pricelist_item_id = source.pricelist_item_id

WHEN MATCHED
AND target.updated_at < source.updated_at

THEN UPDATE SET
    target.pricelist_id = source.pricelist_id,
    target.company_id = source.company_id,
    target.product_tmpl_id = source.product_tmpl_id,
    target.product_id = source.product_id,
    target.categ_id = source.categ_id,
    target.applied_on = source.applied_on,
    target.compute_price = source.compute_price,
    target.fixed_price = source.fixed_price,
    target.percent_price = source.percent_price,
    target.price_discount = source.price_discount,
    target.price_surcharge = source.price_surcharge,
    target.price_markup = source.price_markup,
    target.price_min_margin = source.price_min_margin,
    target.price_max_margin = source.price_max_margin,
    target.valid_from = source.valid_from,
    target.valid_to = source.valid_to,
    target.created_at = source.created_at,
    target.updated_at = source.updated_at,
    target.dwh_loaded_at = current_timestamp()

WHEN NOT MATCHED THEN
INSERT
(
    pricelist_item_id,
    pricelist_id,
    company_id,
    product_tmpl_id,
    product_id,
    categ_id,
    applied_on,
    compute_price,
    fixed_price,
    percent_price,
    price_discount,
    price_surcharge,
    price_markup,
    price_min_margin,
    price_max_margin,
    valid_from,
    valid_to,
    created_at,
    updated_at,
    dwh_loaded_at
)

VALUES
(
    source.pricelist_item_id,
    source.pricelist_id,
    source.company_id,
    source.product_tmpl_id,
    source.product_id,
    source.categ_id,
    source.applied_on,
    source.compute_price,
    source.fixed_price,
    source.percent_price,
    source.price_discount,
    source.price_surcharge,
    source.price_markup,
    source.price_min_margin,
    source.price_max_margin,
    source.valid_from,
    source.valid_to,
    source.created_at,
    source.updated_at,
    current_timestamp()
);